In [1]:
import pandas as pd 
pd.set_option('display.max_colwidth', None)

In [27]:
import pandas as pd
import numpy as np
import re
import string
from nltk.stem import WordNetLemmatizer

from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    StandardScaler
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.compose import ColumnTransformer
from scipy.sparse import hstack

In [2]:
df=pd.read_csv('../data/aa_dataset-tickets-multi-lang-5-2-50-version.csv')

In [3]:
df.head()

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte einen gravierenden Sicherheitsvorfall melden, der gegenwärtig mehrere Komponenten unserer Infrastruktur betrifft. Betroffene Geräte umfassen Projektoren, Bildschirme und Speicherlösungen auf Cloud-Plattformen. Der Grund für die Annahme ist, dass der Vorfall eine potenzielle Datenverletzung im Zusammenhang mit einer Cyberattacke darstellt, was ein erhebliches Risiko für sensible Informationen und den laufenden Geschäftsbetrieb unserer Organisation bedeutet.\n\nUnsere initialen Untersuchungen haben ungewöhnliche Aktivitäten und Abweichungen bei den Geräten ergeben. Trotz der Umsetzung unserer standardisierten Behebungs- und Eindämmungsmaßnahmen konnte die Bedrohung bislang nicht vollständig eliminiert.","Vielen Dank für die Meldung des kritischen Sicherheitsvorfalls und die Bereitstellung der Übersicht über die betroffenen Geräte sowie der ergriffenen ersten Maßnahmen. Wir erkennen die Dringlichkeit und Schwere der Lage an und setzen alles daran, den Fall prioritär zu bearbeiten. Für eine umgehende Untersuchung benötigen wir zusätzliche Informationen: Bitte senden Sie uns spezifische Protokolle der betroffenen Projektoren, Bildschirme und Cloud-Speichersysteme, inklusive Zeitstempel verdächtiger Aktivitäten sowie ungewöhnlicher Fehlermeldungen. Falls möglich, fügen Sie auch eine Zusammenfassung der bereits durchgeführten Maßnahmen bei.",Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?","Thank you for reaching out, <name>. We are aware of the outage affecting the centralized account management system, and our technical team is actively working to resolve the issue. In the meantime, we suggest using alternative methods to manage your account, with a focus on restoring service as quickly as possible. We will provide an update as soon as the service is back online. We apologize for the inconvenience and appreciate your patience. If you have any further questions, please let us know.",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Features,"Dear Customer Support Team,\n\nI hope this message reaches you well. I am reaching out to request detailed information about the capabilities of your smart home integration products listed on your website. As a potential customer aiming to develop a seamlessly interconnected home environment, it is essential to understand how your products interact with various smart home platforms.\n\nCould you kindly provide detailed compatibility information with popular smart home ecosystems such as Amazon Alexa, Google Assistant, and Apple?","Thank you for your inquiry. Our products support integration with Amazon Alexa, Google Assistant, and Apple HomeKit. Compatibility details can differ depending on the specific item; please let us know which models you are interested in. The setup process is generally user-friendly but may require professional installation. We regularly update our software to provide enhanced features. For comprehensive information on compatibility with upcoming updates, please specify the models you are considering.",Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Rega

In [4]:
df['type'].value_counts()

type
Incident    11466
Request      8187
Problem      6012
Change       2922
Name: count, dtype: int64

In [3]:
df=pd.read_csv('../data/ticket_classification.csv')
df=df.dropna()

In [4]:

def clean_text(text):
    lemmatizer= WordNetLemmatizer()
    stop_words=set(stopwords.words('english'))
    # Lowercase
    text = text.lower()

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    words = text.split()

    # Remove stopwords + lemmatization
    cleaned_words = []

    for word in words:

        # Remove stopwords
        if word not in stop_words:

            # Lemmatize
            lemma_word = lemmatizer.lemmatize(word)

            cleaned_words.append(lemma_word)

    # Join back
    text = " ".join(cleaned_words)

    return text

In [5]:
# type_map={
#     'Problem':1,
#     'Request':0,
#     'Change':2,
#     'Incident':3
# }

In [6]:
# df['type']=df['type'].map(type_map)

In [7]:
df['Full Text'] = df['type'].apply(clean_text)+" "+df['subject'].apply(clean_text)+" "+df['body'].apply(clean_text)

# Let's see what a row looks like before and after
print("BEFORE:\n", df['body'].iloc[0])
print("\nAFTER:\n", df['Full Text'].iloc[0])

BEFORE:
 Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?

AFTER:
 incident account disruption dear customer support teamnni writing report significant problem centralized account management portal currently appears offline outage blocking access account setting leading substantial inconvenience attempted log multiple time using different browser device issue persistsnncould please provide update outage status estimated time resolution also alternative way access manage account downtime


In [8]:
# df['char_count'] = df['Full Text'].apply(len)
# df['word_count'] = df['Full Text'].apply(
#     lambda x: len(x.split())
# )
# df['avg_word_length'] = df['Full Text'].apply(
#     lambda x: np.mean([len(word) for word in x.split()])
#     if len(x.split()) > 0 else 0
# )

In [9]:
df['priority'].value_counts()

priority
medium    10130
high       9454
low        5033
Name: count, dtype: int64

In [10]:
target_map={
    'high':2,
    'medium':1,
    'low':0
}
df['target']=df['priority'].map(target_map)

In [11]:
df['target'].value_counts()

target
1    10130
2     9454
0     5033
Name: count, dtype: int64

In [12]:
# categorical_features = ['type']

# numerical_features = [
#     'char_count',
#     'word_count',
#     'avg_word_length',
# ]

# text_feature = 'Full Text'

In [13]:
# tfidf = TfidfVectorizer(max_features=10000,ngram_range=(1,2),
#                         min_df=2,
#                         max_df=0.95)
# X_text = tfidf.fit_transform(df[text_feature])

In [14]:
# preprocessor = ColumnTransformer(
#     transformers=[

#         (
#             'cat',
#             OneHotEncoder(handle_unknown='ignore'),
#             categorical_features
#         ),

#         (
#             'num',
#             StandardScaler(),
#             numerical_features
#         )
#     ]
# )

# X_tabular = preprocessor.fit_transform(
#     df[categorical_features + numerical_features]
# )

In [15]:
# X = hstack([X_text, X_tabular])
# y = df['target']

In [16]:
# y.isnull().sum()

In [17]:
tfidf = TfidfVectorizer(max_features=10000,ngram_range=(1,2),
                        min_df=2,
                        max_df=0.95)
X_text = tfidf.fit_transform(df['Full Text'])

In [18]:
X = X_text
y = df['target']

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
random_state=42,stratify=y)

In [20]:
# tfidf = TfidfVectorizer(max_features=10000,ngram_range=(1,2),
#                         min_df=2,
#                         max_df=0.95)
# X_train_tfidf = tfidf.fit_transform(X_train)
# X_test_tfidf = tfidf.transform(X_test)

In [21]:
X_train=X_train.astype('float32', copy=False)
X_test=X_test.astype('float32', copy=False)

In [22]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
log=LogisticRegression(
    max_iter=2000,
    class_weight='balanced'
)
log.fit(X_train,y_train)
predictions=log.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score (Weighted): {f1_weighted:.4f}")
print(f"F1-Score (Macro): {f1_macro:.4f}\n")

Accuracy: 0.6095
F1-Score (Weighted): 0.6114
F1-Score (Macro): 0.6019



In [23]:
dt_model = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_model.fit(X_train, y_train)
    
    # Predict
predictions = dt_model.predict(X_test)
# text_predictions = target_encoder.inverse_transform(predictions)
# text_y_test = target_encoder.inverse_transform(y_test)
# print(f"---Results ---")
# print(f"Accuracy: {accuracy:.4f}")
# print(f"F1-Score (Weighted): {f1_weighted:.4f}")
# print(f"F1-Score (Macro): {f1_macro:.4f}\n")
# print(classification_report(text_y_test, text_predictions, zero_division=0))

accuracy = accuracy_score(y_test, predictions)
f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score (Weighted): {f1_weighted:.4f}")
print(f"F1-Score (Macro): {f1_macro:.4f}\n")

Accuracy: 0.4760
F1-Score (Weighted): 0.4113
F1-Score (Macro): 0.3556



In [24]:
from sklearn.naive_bayes import MultinomialNB
nb_model = MultinomialNB(alpha=0.001)
nb_model.fit(X_train, y_train)
    
# Predict
predictions = nb_model.predict(X_test)
# text_predictions = target_encoder.inverse_transform(predictions)
# text_y_test = target_encoder.inverse_transform(y_test)
    
    # Calculate evaluation metrics
accuracy = accuracy_score(y_test, predictions)
f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)
print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score (Weighted): {f1_weighted:.4f}")
print(f"F1-Score (Macro): {f1_macro:.4f}\n")
# print(classification_report(text_y_test, text_predictions, zero_division=0))

Accuracy: 0.5603
F1-Score (Weighted): 0.5497
F1-Score (Macro): 0.5234



In [25]:

from xgboost import XGBClassifier
xgb_model = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42
)

xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_test)

print("=" * 50)
print("XGBOOST RESULTS")
print("=" * 50)

print("Accuracy:",
      accuracy_score(y_test, xgb_preds))

print("F1 Score:",
      f1_score(y_test, xgb_preds, average='macro'))

print(classification_report(y_test, xgb_preds))

XGBOOST RESULTS
Accuracy: 0.6383021933387489
F1 Score: 0.5879235449180366
              precision    recall  f1-score   support

           0       0.78      0.28      0.41      1007
           1       0.58      0.80      0.68      2026
           2       0.69      0.66      0.67      1891

    accuracy                           0.64      4924
   macro avg       0.68      0.58      0.59      4924
weighted avg       0.67      0.64      0.62      4924



In [26]:
from lightgbm import LGBMClassifier
lgbm_model = LGBMClassifier(
    objective='multiclass',
    random_state=42
)

lgbm_model.fit(X_train, y_train)

lgbm_preds = lgbm_model.predict(X_test)

print("=" * 50)
print("LIGHTGBM RESULTS")
print("=" * 50)

print("Accuracy:",
      accuracy_score(y_test, lgbm_preds))

print("F1 Score:",
      f1_score(y_test, lgbm_preds, average='macro'))

print(classification_report(y_test, lgbm_preds))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.435447 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 253915
[LightGBM] [Info] Number of data points in the train set: 19693, number of used features: 6924
[LightGBM] [Info] Start training from score -1.587490
[LightGBM] [Info] Start training from score -0.887905
[LightGBM] [Info] Start training from score -0.956995
LIGHTGBM RESULTS
Accuracy: 0.6228675873273761
F1 Score: 0.578760342327506
              precision    recall  f1-score   support

           0       0.70      0.30      0.42      1007
           1       0.58      0.75      0.66      2026
           2       0.66      0.66      0.66      1891

    accuracy                           0.62      4924
   macro avg       0.65      0.57      0.58      4924
weighted avg       0.64      0.62      0.61      4924



/Users/rishabh/miniconda3/envs/DL_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
#=========================================================
# XGBOOST RANDOM SEARCH
# =========================================================

xgb_model = XGBClassifier(
    objective='multi:softprob',   # use 'binary:logistic' for binary classification
    eval_metric='mlogloss',
    min_child_weight=3,
    random_state=42,
    max_depth=10,
    subsample=1,
    gamma=0.1,
    reg_lambda=3,




)

xgb_param_grid = {
    'n_estimators': [350,400,450],
    'learning_rate': [0.4,0.5],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [5,7],
    'reg_alpha': [0.01,0.001],
}

xgb_random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_param_grid,
    n_iter=50,
    scoring='accuracy',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    n_estimtors=500,
    reg_lambda=5,
    reg_alpha=0,

)

xgb_random_search.fit(X_train, y_train)

print("\n========== XGBOOST BEST PARAMETERS ==========")
print(xgb_random_search.best_params_)

# Best Model
best_xgb = xgb_random_search.best_estimator_

# Predictions
xgb_preds = best_xgb.predict(X_test)

print("\n========== XGBOOST RESULTS ==========")
print("Accuracy:", accuracy_score(y_test, xgb_preds))
print(classification_report(y_test, xgb_preds))

Fitting 5 folds for each of 48 candidates, totalling 240 fits


/Users/rishabh/miniconda3/envs/DL_env/lib/python3.12/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 48 is smaller than n_iter=50. Running 48 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[CV] END colsample_bytree=0.8, learning_rate=0.4, min_child_weight=5, n_estimators=350, reg_alpha=0.001; total time=13.0min
[CV] END colsample_bytree=0.8, learning_rate=0.4, min_child_weight=5, n_estimators=350, reg_alpha=0.01; total time=13.0min
[CV] END colsample_bytree=0.8, learning_rate=0.4, min_child_weight=5, n_estimators=350, reg_alpha=0.01; total time=13.1min
[CV] END colsample_bytree=0.8, learning_rate=0.4, min_child_weight=5, n_estimators=350, reg_alpha=0.001; total time=13.1min
[CV] END colsample_bytree=0.8, learning_rate=0.4, min_child_weight=5, n_estimators=350, reg_alpha=0.01; total time=13.1min
[CV] END colsample_bytree=0.8, learning_rate=0.4, min_child_weight=5, n_estimators=350, reg_alpha=0.01; total time=13.1min
[CV] END colsample_bytree=0.8, learning_rate=0.4, min_child_weight=5, n_estimators=350, reg_alpha=0.01; total time=13.1min
[CV] END colsample_bytree=0.8, learning_rate=0.4, min_child_weight=5, n_estimators=350, reg_alpha=0.001; total time=13.3min
[CV] END cols

KeyboardInterrupt: 

{'subsample': 1.0, 'reg_lambda': 5, 'reg_alpha': 0, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 10, 'learning_rate': 0.2, 'gamma': 0.1, 'colsample_bytree': 1.0}
Accuracy: 0.7300974817221771